# TRACER Algorithmic Changes

What changed in the TRACER algorithm between the legacy workflow on `feature/relative-stitching` (the parent of this branch) and the production workflow now on `optimization/core-refactor`. Each section describes one concern, links to the source file/function and the commit that introduced the change, and (where measured) shows the bench numbers that justified it.

If you want to see either workflow run end-to-end, see:
- [`legacy_workflow.ipynb`](legacy_workflow.ipynb) — the pre-this-branch pipeline
- [`segmented_workflow.ipynb`](segmented_workflow.ipynb) — the modern segmented pipeline
- [`noseg_workflow.ipynb`](noseg_workflow.ipynb) — the modern no-segmentation variant

## Index of changes

1. [Bootstrap NPMI panel](#1-bootstrap-npmi-panel)
2. [PMI as the primary metric](#2-pmi-as-the-primary-metric)
3. [Grid-bin graph primitives](#3-grid-bin-graph-primitives)
4. [Same-bin Stage 2 semantic](#4-same-bin-stage-2-semantic)
5. [Stage 4 position move](#5-stage-4-position-move)
6. [Two stitching passes → one](#6-two-stitching-passes--one)
7. [Pre-Stage-2 rescue (Initial Rescue)](#7-pre-stage-2-rescue-initial-rescue)
8. [Mean-PMI rescue veto](#8-mean-pmi-rescue-veto)
9. [Shield-label absorption fix](#9-shield-label-absorption-fix)
10. [Single canonical output column](#10-single-canonical-output-column)
11. [Project-folder convention (`tracer.data`)](#11-project-folder-convention-tracerdata)

## Quick repository links

- Branch: [`optimization/core-refactor`](https://github.com/imlong4real/TRACER/tree/optimization/core-refactor)
- Commit log on this branch: `git log feature/relative-stitching..optimization/core-refactor`

---

## 1. Bootstrap NPMI panel

**Function**: [`tracer.metrics.compute_npmi_bootstrap`](../../src/tracer/metrics.py)

**What changed**: The legacy NPMI panel was the file `<project>/data/<name>_npmi.csv`, computed via `tracer.metrics.compute_npmi`. That builder used observation-only NaN handling: if a gene-pair was never co-observed in the same `cell_id`, its NPMI entry was NaN — *regardless* of how many transcripts each gene had individually. So a pair with 1,000 individual transcripts of each gene but zero co-observations got the same `NaN` as a pair with 5 of each. Both treated identically as "no info."

**The new bootstrap policy** (added in earlier session work, then made the default this branch) uses *expected co-occurrence under independence* as the evidence floor:

- A pair is in the **explicit-evidence tier** if its expected co-occurrence count under independence (`E[C_ij] = N · P_i · P_j`) exceeds `min_expected_cooccur_for_evidence` (default 10).
- A pair with high expected co-occurrence but zero observed gets a **`neg_one` sentinel**, i.e. "this pair systematically avoids — strong negative evidence," instead of NaN.
- A pair with low expected co-occurrence and no observed stays at NaN.

**Why it matters**: Stage 1 prune and the rescue veto operate on the panel. Under the legacy NaN policy, ~700 high-evidence avoidance pairs were silently invisible — the prune saw them as "no info" and didn't drop the gene-incoherent transcripts. Under the bootstrap policy, those pairs hit the prune as strong negative evidence.

**Where to use**: bench it via `--npmi-source bootstrap` (the default). The panel is cached at `<project>/data/npmi_bs*.csv` after first compute.

## 2. PMI as the primary metric

**What changed**: Legacy used **NPMI** (normalized PMI, bounded to [-1, 1]) throughout. Modern workflow uses **PMI** (unbounded, log-ratio of joint vs marginal probability). The bench wires this via `--metric pmi`.

**Why**:
- PMI's unbounded scale means strong co-association doesn't saturate. NPMI of +0.99 vs +0.999 are very different evidence levels but compress similarly; PMI distinguishes them.
- Threshold semantics are more natural: `PMI > ln(1.5)` ≡ "≥ 50 % above independence"; `PMI < ln(1/3)` ≡ "≤ 1/3 of independence." These are direct probabilistic statements.
- The same `compute_npmi_bootstrap` builder produces either by setting `metric="pmi"`.

**Threshold mapping**: legacy NPMI values map to PMI by the factor `ln(1.5) / 0.05 ≈ 8.103`:
- legacy NPMI prune `−0.05` → PMI scale `−0.405`
- modern PMI prune `+ln(1.5)` ≈ `+0.405` (note: sign and direction flipped — *strict positive evidence required* instead of *strong negative evidence to drop*)
- modern rescue negative threshold `ln(1/3)` ≈ `−1.099`

The sign flip on the prune is the bigger conceptual change: legacy's `-0.05` was a *lenient veto* (drop only if strongly avoided); modern's `+ln(1.5)` is a *strict positive requirement* (the gene must be measurably co-coherent with the cell's panel).

## 3. Grid-bin graph primitives

**Functions**: [`tracer.graph.build_grid_graph_xy`](../../src/tracer/graph.py) and [`tracer.graph.build_grid_graph_xyz`](../../src/tracer/graph.py), introduced in commit [`abb15ac`](https://github.com/imlong4real/TRACER/commit/abb15ac).

**What changed**: Legacy Stage 2 (Group) used kNN(k=8, d≤1.5 µm) via `cKDTree`. Legacy Stage 4 (Split) used kNN(k=5, d≤5 µm). Both have a fundamental issue in dense tissue: a transcript's k nearest neighbors are mostly in **other** cells, so kNN-restricted-to-same-label connectivity artificially fragments compact cells.

**The new graph primitives** bin transcripts into a regular grid and use bin-adjacency as the connectivity:

- **`build_grid_graph_xy`** (2D): bin tx by xy at scale G; emit edges among same-bin transcripts plus 4- or 8-adjacent-bin neighbors. Optional Euclidean post-filter.
- **`build_grid_graph_xyz`** (3D): xy 8-conn × z bins at depth ±1. 21-bit-per-axis int64 packed key. The Stage 4 production primitive on Xenium z-aware data.
- A new **`neighborhood="self"`** option on `build_grid_graph_xy` emits same-bin edges only (no cross-bin), enabling the ["1 bin = 1 candidate cell" semantic](#4-same-bin-stage-2-semantic) at Stage 2.

**Why this is better**: the graph topology is determined by spatial proximity at known scale (the bin size), not by k-nearest-density. Same-label tx in adjacent bins are guaranteed to be candidate edges, regardless of how many other-label tx happen to live in between.

**Visium HD compatibility**: under Visium HD, `(x, y)` are already bin centers and there's no z. `build_grid_graph_xy(neighborhood="self", exact_filter=False)` reduces to "each bin's transcripts form a clique" — exactly the right primitive. No fine-coordinate dependency.

## 4. Same-bin Stage 2 semantic

**Where it's used**: `annotate_unassigned_components_fast` configured with `build_grid_graph_xy(G=8, neighborhood="self", exact_filter=False)` and `min_comp_size=1`.

**What changed**: Legacy Stage 2 used a kNN graph at d≤1.5 µm and `min_comp_size=10`. The 1.5 µm reach is shorter than typical cell diameters (3-10 µm), so a single biological cell often had its unassigned transcripts split across multiple connected components. The `min_comp_size=10` filter then dropped many of those small fragments before they could be merged later.

**The new semantic**: every G=8 µm × 8 µm xy bin's transcripts form one component. Cross-bin merging is deferred to Stage 5 (Stitch) where it belongs. `min_comp_size=1` defers all size filtering to the demote pass.

**Rationale**: Stage 2's job is to propose *candidates* for cells. The right semantic is "each bin is one candidate." Whether a candidate survives — and whether two candidates should merge — is Stitch's job, not Stage 2's.

**Bench-validated impact** at full data: ~+8 % more partials (more candidates surviving Stage 2 → Stitch can merge or demote them downstream), with neutral coverage and partial coherence.

## 5. Stage 4 position move

**Function**: [`tracer.spatial.enforce_spatial_coherence_fast`](../../src/tracer/spatial.py) — same function, called at a different position in the pipeline.

**What changed**: Legacy ran Stage 4 BETWEEN the two stitching passes (after Stage 3 "initial Stitch," before Stage 5 "final Stitch"). Modern runs it RIGHT AFTER Stage 1 prune.

**Why**:
- The only labels that can be spatially disconnected at this point are the *input* `cell_id` labels (input segmentation defects: Mickey-Mouse cells, edge effects).
- Stage 2's same-bin components are spatially compact by construction (bounded to one G=8 µm bin).
- Stage 5's grid-gated stitch produces spatially-compact merges (gated at G=2 µm).
- So one Stage 4 pass at the start is sufficient — eliminates the need for the second pass between two stitches.

**The bench position is configured via `--stage4-position after-s1`** (modern) vs `after-annotate` (legacy).

## 6. Two stitching passes → one

**Function**: [`tracer.stitching.apply_stitching_to_transcripts_memory_efficient`](../../src/tracer/stitching.py)

**What changed**: Legacy ran Stitch twice — once after Group (Stage 3), once after Split (Stage 5). The intuition was that Split might produce fragments that benefit from a second merge pass.

**Why dropped**: with Stage 4 moved to right-after-Stage 1, the only entities Stitch sees are (a) survivors of input segmentation + Stage 4, and (b) Stage 2's bin-bounded components. These don't fragment further between Group and Stitch, so a second pass isn't doing useful work — it's just reapplying the same merge rule on the same candidates.

**Other stitch changes on this branch**:
- `purity_threshold` parameter renamed to `threshold` in the modern API.
- `use_3d=True` flag is gone — 3D handling is inferred from `coord_cols`.
- `deltaC_min=0.01` (legacy strict) → `0.0` (modern: any non-negative ΔC merge accepted; `penalize_simplicity=True` handles the size penalty).
- `candidate_source="grid"` at G=2 µm 8-conn (modern default) — replaces legacy implicit kNN candidates.

## 7. Pre-Stage-2 rescue (Initial Rescue)

**Function**: [`tracer.spatial.pre_stage2_rescue`](../../src/tracer/spatial.py) — added in commit [`6aa3f3e`](https://github.com/imlong4real/TRACER/commit/6aa3f3e).

**What changed**: Legacy had no rescue between Stage 1 (Prune) and Stage 2 (Group). All unassigned transcripts went straight from Prune output into Group. Some of those were biologically real members of nearby cells — they were just gene-coherence outliers under their original cell's panel. Group then clustered them into UNASSIGNED_* components, and Phase 6 reassigned them by nearest-entity (no NPMI veto).

**The new Initial Rescue**: between Prune and Group, distance-priority rescue with two new mechanisms:
1. **Cluster guard**: bin unassigned tx at G=2 µm; tx with ≥3 same-bin same-gene peers are *shielded* from absorption (kept as `-1`). The shield protects gene-coherent unassigned clusters that should later become their own UNASSIGNED_* component in Group.
2. **Mean-PMI veto**: see [section 8](#8-mean-pmi-rescue-veto).

Net effect: tx that legacy left for Group + Phase 6 to handle now get attached to compatible neighbors earlier when there's clear evidence they belong; tx that have *their own* coherent cluster of peers stay as `-1` so Group can find them.

## 8. Mean-PMI rescue veto

**Functions**: [`tracer.spatial.reassign_unassigned_grid_pool`](../../src/tracer/spatial.py) (post-Stitch rescue), [`tracer.spatial.pre_stage2_rescue`](../../src/tracer/spatial.py) (Initial Rescue) — both gained `veto_mode="mean"`, `mean_threshold=0.0`, `small_entity_guard_n=0` parameters in commit [`6aa3f3e`](https://github.com/imlong4real/TRACER/commit/6aa3f3e).

**What changed**: Legacy used `reassign_unassigned_to_nearby_entities` with **no NPMI veto** — picked the nearest entity within `dist_threshold=5.0` µm regardless of gene compatibility. A known limitation: macrophage transcripts could be absorbed into adjacent epithelial cells if proximity was tight enough, even when the gene-pair PMI was strongly negative.

**The new veto rule**:
- For each candidate (tx, entity) pair, compute the mean PMI between the tx's gene `g` and each gene `g'` in the entity's panel (only over observed-PMI pairs; missing pairs are skipped from the mean).
- Reject the candidate entity if `mean(PMI) ≤ 0` — i.e. on average the gene is no more associated than independence with the entity's panel.
- Among the surviving entities, pick the closest one by min-tx Euclidean distance.

**Why mean instead of min**: an earlier version used `min(PMI) ≤ neg_threshold` (any single strongly-negative pair vetoes). That was overly sensitive to *complementary* state markers (e.g. AT2 / macrophage co-markers with mildly negative pairwise PMI), which would veto a rescue even when the aggregate evidence was strongly positive. The mean rule integrates evidence across all observed pairs and tolerates a single outlier.

**`small_entity_guard_n=0`**: when `mean` mode is active, an entity with fewer than `small_entity_guard_n` observed PMI pairs falls back to the `min` rule (mean is too noisy on small samples). Setting it to 0 disables the fallback entirely — every observed negative-PMI pair is real evidence.

## 9. Shield-label absorption fix

**Bug fix in commit [`6aa3f3e`](https://github.com/imlong4real/TRACER/commit/6aa3f3e)** at [`tracer.spatial.pre_stage2_rescue`](../../src/tracer/spatial.py).

**The bug**: `pre_stage2_rescue` shields cluster-guarded unassigned tx by relabeling them to the literal string `"__GUARD_SKIP__"` before invoking `reassign_unassigned_grid_pool`, then restoring them to `-1` after. But the rescue function saw `__GUARD_SKIP__` as just another assigned-entity label and let surrounding unassigned tx be absorbed *into* it under the distance-priority rule. The post-rescue restore only reverted the originally-shielded tx, leaving the bogus absorptions stuck on the shield label.

**Visibility**:
- In normal segmented runs: rare and silent — real cells were almost always closer than the shielded clusters, so most rescued tx went to real cells.
- Under no-segmentation runs (`--vhd-degrade no-seg`): the shield label became the *only* "assigned entity" present, and the rescue absorbed thousands of tx into it. Manifested as a phantom `1 cell, ~5000 tx` that appeared from nothing.

**The fix**: after the rescue, find ALL tx still labeled `__GUARD_SKIP__` (originally shielded + bogus absorptions), revert them to `-1`, and subtract the absorbed-count from `n_reassigned`. New stats key `n_absorbed_into_shield_reverted` exposes the count for diagnostics.

## 10. Single canonical output column

**Commit**: [`fd1dd73`](https://github.com/imlong4real/TRACER/commit/fd1dd73) — refactor(api): collapse 7 cell_id_* columns into single tracer_id.

**What changed**: Legacy wrote 7 different columns through the pipeline:

| legacy column | written by |
|---|---|
| `cell_id_npmi_cons_p1` | Stage 1 pass 1 |
| `npmi_cons_p1_status` | Stage 1 pass 1 status |
| `cell_id_npmi_cons_p2` | Stage 1 pass 2 |
| `npmi_cons_p2_status` | Stage 1 pass 2 status |
| `cell_id_final` | Stage 2 |
| `cell_id_stitched` | Stage 3 / Stage 5 stitch |
| `cell_id_spatial` | Stage 4 split |
| `cell_id_finetuned` | Stage 5 stitch (final) |
| `cell_id_finetuned_2` | Phase 6 reassignment |

**Modern**: a single `tracer_id` column that each stage writes to in place. Stage 5 writes its own column `stitched`. Snapshot columns are gated behind `debug_stages=True`. Downstream code that read `cell_id_finetuned_2` should now read `tracer_id` (or `stitched` after Stage 5).

## 11. Project-folder convention (`tracer.data`)

**New module**: [`tracer.data`](../../src/tracer/data.py), commit [`70bc508`](https://github.com/imlong4real/TRACER/commit/70bc508).

**What changed**: Legacy tutorials and bench scripts each had their own ad-hoc parquet/NPMI-CSV path resolution. The new `tracer.data` module establishes a project-folder convention:

```
<project>/
  data/
    <name>.parquet         # input transcript table
    npmi_bs*.csv           # bootstrap PMI cache (optional)
```

Two functions:
- `discover_data_files(project_dir) → (parquet_path, npmi_cache_path | None)`
- `load_full_df(project_dir=..., parquet_path=...) → DataFrame`

The package itself remains DataFrame-in / DataFrame-out — file IO is still the caller's responsibility — but tutorials, bench, and downstream scripts can now share the same convention.

---

## Validated impact at full data (1.44M tx, 58,405 input cells)

| metric | legacy | modern segmented |
|---|---:|---:|
| within-cell fragmentation | ~16% (Stage 4 in middle position lets Mickey-Mouse cells through) | ~5% |
| coverage | ~96% | 96.3% |
| traveler mis-placement | higher (no NPMI veto on rescue) | ~10-14% |
| output columns | 7 | 1-2 |
| stitch passes | 2 | 1 |
| VHD compatible | no (kNN needs fine coords) | yes (grid-bin only) |

Per-component coherence (count-based, PMI > ln(1.5)) on segmented variants is essentially unchanged across Stage 4 algorithm choices (~0.83-0.87 cell-mean), confirming the algorithmic refinements operate primarily at the *partition* level — fewer Mickey-Mouse cells, fewer mis-placed transcripts — without sacrificing per-entity coherence quality.

## Branch summary

Total commits on `optimization/core-refactor` vs main: **12**.

Commits making algorithmic changes (this branch's contributions, not from upstream):

| commit | concern |
|---|---|
| [`abb15ac`](https://github.com/imlong4real/TRACER/commit/abb15ac) | feat(graph): grid-bin graph builders + same-bin neighborhood option |
| [`6aa3f3e`](https://github.com/imlong4real/TRACER/commit/6aa3f3e) | fix(spatial): pre_stage2_rescue shield-label leak + rescue/Stage-4 overhaul |
| [`70bc508`](https://github.com/imlong4real/TRACER/commit/70bc508) | feat(data): tracer.data module for project-folder data loading |

Other commits on the branch are infrastructure (build, refactor, perf):

| commit | concern |
|---|---|
| [`fd1dd73`](https://github.com/imlong4real/TRACER/commit/fd1dd73) | refactor(api): collapse 7 cell_id_* columns into single tracer_id |
| [`32e0d1d`](https://github.com/imlong4real/TRACER/commit/32e0d1d) | perf(memory): categorical dtypes + in_place kwargs |
| [`ba2ce15`](https://github.com/imlong4real/TRACER/commit/ba2ce15) | refactor(core): split core.py into modules |
| [`8e53161`](https://github.com/imlong4real/TRACER/commit/8e53161) | refactor(metrics): sparse CSR + numba kernel for purity/conflict |
| [`2522f45`](https://github.com/imlong4real/TRACER/commit/2522f45) | refactor: Cython hard dependency; drop runtime fallbacks |
| [`7ad86d2`](https://github.com/imlong4real/TRACER/commit/7ad86d2) | build: compile cython extensions at install time |
| [`e812629`](https://github.com/imlong4real/TRACER/commit/e812629) | fix(spatial): vectorize reassignment loop |
| [`a2e1f37`](https://github.com/imlong4real/TRACER/commit/a2e1f37) | merge-base — Update core stitching logic for relative mode (parent state) |